In [ ]:
# Berlin Food Delivery ETA Prediction
## Step 3: Feature Engineering

Goal: Turn raw cleaned data into model-ready features:
- Real distance from coordinates (haversine)
- Datetime-derived features (hour, day of week, rush hour)
- Prep time before pickup
- Encoded categorical variables

In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv('train_cleaned.csv')
test = pd.read_csv('test_cleaned.csv')

train.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,Sunny,High,2,Snack,motorcycle,0.0,No,Urban,24.0
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,Stormy,Jam,2,Snack,scooter,1.0,No,Metropolitian,33.0
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,Sandstorms,Low,0,Drinks,motorcycle,1.0,No,Urban,26.0
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,Sunny,Medium,0,Buffet,motorcycle,1.0,No,Metropolitian,21.0
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,Cloudy,High,1,Snack,scooter,1.0,No,Metropolitian,30.0


In [3]:
## Haversine distance function

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

for df in [train, test]:
    df['distance_km'] = haversine_distance(
        df['Restaurant_latitude'], df['Restaurant_longitude'],
        df['Delivery_location_latitude'], df['Delivery_location_longitude']
    )

train['distance_km'].describe()

count    45593.000000
mean        99.303911
std       1099.731281
min          1.465067
25%          4.663493
50%          9.264281
75%         13.763977
max      19692.674606
Name: distance_km, dtype: float64

In [4]:
## checking for invalid distances

# Some rows had unrealistic distances (e.g. 0 km or 1000+ km due to bad coordinates)
print(train['distance_km'].sort_values(ascending=False).head(10))
print(train[train['distance_km'] < 0.5].shape[0], "rows under 0.5km")

33533    19692.674606
6788     19688.001288
2484     19683.687561
18826    19677.180552
9535     19070.408110
762      19070.337839
35535    19069.158946
22291    19068.246962
30633    19067.128547
43454    19066.150742
Name: distance_km, dtype: float64
0 rows under 0.5km


In [6]:
## handled distance outliers here

# Caped unrealistic distances at a sensible delivery-range max (e.g. 30km for a city)
distance_cap = 30
train['distance_km'] = train['distance_km'].clip(upper=distance_cap)
test['distance_km'] = test['distance_km'].clip(upper=distance_cap)

In [7]:
## parsed datetime features

for df in [train, test]:
    df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%d-%m-%Y')
    df['Time_Orderd'] = pd.to_datetime(df['Time_Orderd'], format='%H:%M:%S', errors='coerce').dt.time
    df['Time_Order_picked'] = pd.to_datetime(df['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.time

    df['order_hour'] = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce').dt.hour
    df['day_of_week'] = df['Order_Date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_rush_hour'] = df['order_hour'].isin([12, 13, 19, 20]).astype(int)

train[['order_hour', 'day_of_week', 'is_weekend', 'is_rush_hour']].head()

/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_91094/4240260406.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['order_hour'] = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce').dt.hour
/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_91094/4240260406.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['order_hour'] = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce').dt.hour


,order_hour,day_of_week,is_weekend,is_rush_hour
0,11.0,5,1,0
1,19.0,4,0,1
2,8.0,5,1,0
3,18.0,1,0,0
4,13.0,5,1,1


In [8]:
## preparation time before pickup (time between order placed and picked up)

for df in [train, test]:
    order_dt = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce')
    pickup_dt = pd.to_datetime(df['Time_Order_picked'].astype(str), errors='coerce')
    df['prep_time_min'] = (pickup_dt - order_dt).dt.total_seconds() / 60
    df['prep_time_min'] = df['prep_time_min'].apply(lambda x: x + 1440 if x < 0 else x)  # handle midnight rollover

train['prep_time_min'].describe()


/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_91094/4035421302.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_dt = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce')
/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_91094/4035421302.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pickup_dt = pd.to_datetime(df['Time_Order_picked'].astype(str), errors='coerce')
/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_91094/4035421302.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_dt = pd.to_datetime(df['Time_Orderd

count    43862.000000
mean         9.989399
std          4.087516
min          5.000000
25%          5.000000
50%         10.000000
75%         15.000000
max         15.000000
Name: prep_time_min, dtype: float64

In [9]:
## interaction features

# Traffic and weather often compound each other's effect on delay
traffic_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Jam': 4}
train['traffic_level_num'] = train['Road_traffic_density'].map(traffic_map)
test['traffic_level_num'] = test['Road_traffic_density'].map(traffic_map)

train['distance_x_traffic'] = train['distance_km'] * train['traffic_level_num']
test['distance_x_traffic'] = test['distance_km'] * test['traffic_level_num']

train['is_bad_weather'] = train['Weatherconditions'].isin(['Stormy', 'Sandstorms', 'Fog']).astype(int)
test['is_bad_weather'] = test['Weatherconditions'].isin(['Stormy', 'Sandstorms', 'Fog']).astype(int)

In [10]:
## encoded remaining categoricals

categorical_cols = ['Type_of_order', 'Type_of_vehicle', 'City', 'Festival']
train_encoded = pd.get_dummies(train, columns=categorical_cols, drop_first=True)
test_encoded = pd.get_dummies(test, columns=categorical_cols, drop_first=True)

train_encoded.shape, test_encoded.shape

((45593, 34), (11399, 33))

In [11]:
## save feature-engineered data

train_encoded.to_csv('train_features.csv', index=False)
test_encoded.to_csv('test_features.csv', index=False)
print("Saved train_features.csv and test_features.csv")

Saved train_features.csv and test_features.csv
